In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import (
    CarSim,
    Commander,
)
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)
sim = CarSim(prop)
com = Commander(sim)

# プログラムの書き方講座１
## 1.プログラムは命令を上から順番に書く

例えば、命令を日本語で書いてみると・・・
```
まっすぐ2m進め
左に90度回転せよ
まっすぐ2m進め
左に90度回転せよ
まっすぐ2m進め
```
ロボットがどんな動きをするかイメージできるかな？

## 2. プログラム言語では、使える命令が決まっている

さきほどのプログラムを、プログラム言語で書くと・・・
```
run(2.0)
turn(90)
run(2.0)
turn(90)
run(2.0)
```

注：　使える命令は、使用するライブラリ等によって変わる

命令は、
```
関数名(指示値)
```
の形式で書く。上記の例の場合は、runやturnが関数名で、2.0や90が指示値。

関数にどのような指示値が使えるかは、関数によって異なる。指示値は0個や複数個の場合もあるので、よく説明を読んでからプログラムをする必要があるよ。

```
関数名()
関数名(指示値1,指示値2,指示値3)
```

**下のチュートリアルの説明をよく読んで、問題に取り組んでみよう。**

# チュートリアル1

下記の命令を組み合わせてプログラムを書き、ロボットを1m先にあるチェックポイント(goal1)を超えてから、元の位置(goal2)に戻らせよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/s]|move(v=0.2)|
|move|一定時間だけ前に進む|v=速度[m/s], t=時間[s]|move(v=0.2, t=1.0)|
|rotate|一定速度で回転する|w=回転速度[度/s]|rotate(w=90)|
|rotate|一定時間だけ回転する|w=回転速度[度/s], t=時間[s]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ||wait()|

## 注意点
- 命令は直後にwaitを置かない場合、どんどん下に実行されていってしまう。
    - 「一定時間だけ前に進む」「一定時間だけ回転する」を最後までに実行するには、wait命令を直後に置く必要がある
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大10度ほどずれる

In [ ]:
class Tutorial1(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalLine((1.0, 0.0), should_stop=False),
            GoalCircle((0.0, 2.0), 0.2),
        ]
        self.initial_xy = (0.0, 2.0)


MissionDrawer(Tutorial1()).show()
print("最大速度", prop.max_velocity, "m/s")
print("最大回転速度", prop.max_rotate_deg, "度/s")

In [ ]:
move = com.move
rotate = com.rotate
wait = com.wait


class Tutorial1(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalLine((1.0, 0.0), should_stop=False),
            GoalCircle((0.0, 2.0), 0.2),
        ]
        self.initial_xy = (0.0, 2.0)

    def command_func(self):
        ######## ここから下にプログラムを書こう
        move(v=0.2, t=3) #書き方の例
        wait()
        ######## ここより上にプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


sim.set_mission(Tutorial1())
sim.run()
SimDrawer(sim).show()